In [ ]:
# Core
!pip install torch torchvision torchaudio transformers accelerate datasets sentencepiece safetensors pillow evaluate nltk rouge-score qwen_vl_utils bitsandbytes peft

Imports & Config


In [ ]:

import os, random, json, math, time, gc, copy, re, contextlib
from contextlib import nullcontext
from types import SimpleNamespace
from typing import List, Tuple, Dict
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset, DatasetDict, load_from_disk
from PIL import Image

from transformers import (
    AutoProcessor, AutoTokenizer, AutoModelForVision2Seq,
    get_cosine_schedule_with_warmup,
)
from qwen_vl_utils import process_vision_info


def set_seed(seed: int = 42):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ---------------- Config ----------------
CFG = SimpleNamespace(
    # Paths
    model_id="your model",
    out_dir="output_directory",
    swap_cache="swap_cache",
    ds_cache="dataset_cache",

    # Training (Stage A)
    epochs=20,
    num_workers=6,
    lr_ffn=2e-5,          # lower for stability in mimic
    lr_head=5e-5,
    weight_decay=0.05,
    warmup_steps=0,
    warmup_ratio=0.10,    # 10%
    grad_clip=1.0,
    seed=42,
    log_every=200,

    # Batching / memory
    train_micro_batch=10,
    grad_accum=8,
    max_txt_len=256,
    image_short_side=336,

    # Complex-PHM swap (capacity control)
    phm_bases_default=2,      # default B
    top_lang_layers_B3=12,    # set 0 to disable B=3
    include_ffn=True,
    include_misc=False,
    size_threshold=2*262144,
    PROJECT_FROM_DENSE=True,

    # Projector (keep dense during parity)
    FORCE_SWAP_PROJECTOR=False,

    # Residual PHM convex blend + mimic
    PHM_FADE_FRACTION=0.7,      # slower fade (mimic)
    PHM_RECON_WEIGHT=5e-6,      # tiny L2( PHM(x), Dense(x) )
    CE_LABEL_SMOOTH=0.10,
    KD_ENABLE=True,
    KD_T=4.0,
    KD_LAMBDA=0.7,              # max; we ramp 0 -> this over fade window
    KD_EVERY_N=1,               # every step

    # LoRA (TEXT Q/K/V only)
    LORA_TEXT_QKV=True,
    LORA_R=8,
    LORA_ALPHA=16,
    LORA_DROPOUT=0.0,
    LORA_MERGE_ON_SAVE=True,

    # Export compact PHM-only each epoch (for validation)
    EXPORT_COMPACT_ON_SAVE=True,

    # Data prep
    DO_LOAD_FROM_SWAP=False,
    DO_SWAP=True,
    DO_SAVE_SWAP=True,
    DO_DATA_PREP=False,
    DO_SAVE_DATA=True,
    LOPS_SAMPLE_STEPS = 5,
    # # in CFG:
    RESUME_EPOCH = 0,      # start from epoch 1
    RESUME_STEP  = 0,      # start of that epoch
    # Selection metric for best model saved during Stage A
    # choices: "ce_phm" (lower is better) or "cider" (higher is better)
    selection_metric="cider",

    # ---- Stage B (PHM-only finetune) ----
    STAGE_B_ENABLE=True,
    stageB_epochs=10,        # short polish pass (≈ 0.5–1 epoch; integer here)
    stageB_lr=2e-5,
    stageB_wd=0.05,
    stageB_warmup_ratio=0.1,
    stageB_label_smooth=0.10,
    stageB_KD_ENABLE=False,   # optional; default off to keep memory simple
    stageB_KD_T=2.0,
    stageB_KD_LAMBDA=0.2,

    # Val metrics sampling
    VAL_MAX_CAPTION_SAMPLES=400,   # subset for BLEU/METEOR/CIDEr
)

# speed & memory toggles
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True
try: torch.set_float32_matmul_precision("high")
except Exception: pass
os.makedirs(CFG.out_dir, exist_ok=True)


def _fmt_num(n: int) -> str:
    return f"{n/1e9:.2f}B" if n >= 1e9 else (f"{n/1e6:.2f}M" if n >= 1e6 else f"{n:,}")


def count_params(m: nn.Module, label=""):
    tot = sum(p.numel() for p in m.parameters())
    trn = sum(p.numel() for p in m.parameters() if p.requires_grad)
    frz = tot - trn
    if label: print(label)
    print(f"  Total:     {_fmt_num(tot)} ({tot:,})")
    print(f"  Trainable: {_fmt_num(trn)} ({trn:,})")
    print(f"  Frozen:    {_fmt_num(frz)} ({frz:,})")
    return {"total": tot, "trainable": trn, "frozen": frz}


def _uniq_params(params):
    seen = set(); out = []
    for p in params:
        pid = id(p)
        if pid not in seen:
            out.append(p); seen.add(pid)
    return out


LoRA (text Q/K/V only)


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base_linear
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / max(1, r)
        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        if r > 0:
            self.lora_A = nn.Parameter(torch.zeros(r, self.in_features))
            self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)
        else:
            self.register_parameter("lora_A", None)
            self.register_parameter("lora_B", None)
        self.merged = False

    def forward(self, x):
        out = nn.functional.linear(x, self.base.weight, self.base.bias)
        if self.r > 0 and not self.merged:
            after = self.lora_dropout(x) @ self.lora_A.t()
            after = after @ self.lora_B.t()
            out = out + self.scaling * after
        return out

    @torch.no_grad()
    def merge(self):
        if self.merged or self.r == 0: return
        delta = (self.lora_B @ self.lora_A) * self.scaling  # [out, in]
        self.base.weight.add_(delta)
        self.merged = True


@torch.no_grad()
def merge_all_lora_(model: nn.Module, verbose=True):
    for name, mod in model.named_modules():
        if isinstance(mod, LoRALinear):
            mod.merge()
            if verbose: print(f"[lora][merge] {name}")

Complex-PHM + residual wrapper (FFNs)


In [ ]:
class ComplexPHMLinear(nn.Module):
    def __init__(self, din, dout, B=2, bias=True, train_H=False):
        super().__init__()
        assert din % 2 == 0 and dout % 2 == 0, "Use even dims for ComplexPHM"
        self.din, self.dout, self.B = din, dout, B
        self.din2, self.dout2 = din // 2, dout // 2

        # trainable A_b blocks
        self.As = nn.Parameter(torch.randn(B, self.dout2, self.din2) * 0.02)

        # fixed 2x2 bases H_b
        # I, J are the complex bases; for B>=3/4 add K, L to avoid rank issues
        H = torch.zeros(B, 2, 2)
        if B >= 1: H[0].copy_(torch.tensor([[1., 0.],[0., 1.]]))   # I
        if B >= 2: H[1].copy_(torch.tensor([[0.,-1.],[1., 0.]]))   # J
        if B >= 3: H[2].copy_(torch.tensor([[1., 0.],[0.,-1.]]))   # K (diag)
        if B >= 4: H[3].copy_(torch.tensor([[0., 1.],[1., 0.]]))   # L (flip)
        if train_H:
            self.H = nn.Parameter(H)
        else:
            self.register_buffer("H", H)

        self.bias = nn.Parameter(torch.zeros(dout)) if bias else None

    @torch.no_grad()
    def project_from_dense(self, W_dense: torch.Tensor):
        """
        Fit PHM A-bases to a pre-trained dense weight using:
            W_dense ≈ sum_b kron(H[b], A[b])  (2x2 ⊗ dout2×din2)
        For B=2 with canonical H=[I,J], use closed form; else solve least-squares on CPU.
        """
        dout, din = W_dense.shape
        assert dout == self.dout and din == self.din
        dev = W_dense.device

        # reshape dense into [2,2,dout2,din2]
        Wv = (
            W_dense.to(torch.float32)
                   .view(self.dout2, 2, self.din2, 2)
                   .permute(1, 3, 0, 2).contiguous()
        )

        # Fast closed form for canonical B=2 (I,J)
        if self.B == 2:
            Ht = torch.tensor(
                [[[1., 0.],[0., 1.]],   # I
                 [[0.,-1.],[1., 0.]]],  # J
                dtype=torch.float32, device=dev
            )
            if torch.allclose(self.H.to(dev, torch.float32), Ht, atol=1e-6, rtol=1e-6):
                p = Wv[0,0]  # [dout2, din2]
                q = Wv[0,1]
                r = Wv[1,0]
                s = Wv[1,1]
                A0 = 0.5*(p + s)
                A1 = 0.5*(r - q)
                A_est = torch.stack([A0, A1], dim=0)  # [2, dout2, din2]
                self.As.copy_(A_est.to(self.As.device, self.As.dtype))
                return

        # General B: solve (4 x B) @ A_vec = blocks (4 x N), N=dout2*din2
        # Solve on CPU for stability; fall back to pinv if lstsq fails.
        H_flat = self.H.detach().to(torch.float64).view(self.B, 4)   # [B,4]
        A_mat  = H_flat.T.contiguous()                               # [4,B]
        blocks = Wv.reshape(4, self.dout2 * self.din2).to(torch.float64).contiguous()  # [4,N]

        A_cpu = A_mat.cpu()
        B_cpu = blocks.cpu()
        try:
            sol = torch.linalg.lstsq(A_cpu, B_cpu).solution           # [B, N]
        except Exception:
            sol = (torch.linalg.pinv(A_cpu) @ B_cpu)                   # [B, N]

        A_est = sol.view(self.B, self.dout2, self.din2).to(self.As.dtype)
        # guard against NaNs/Infs
        if not torch.isfinite(A_est).all():
            A_est = torch.nan_to_num(A_est, nan=0.0, posinf=0.0, neginf=0.0)

        self.As.copy_(A_est.to(self.As.device))

    def forward(self, x):
        x = x.view(*x.shape[:-1], 2, self.din2)
        xr, xi = x[..., 0, :], x[..., 1, :]
        yr = xr.new_zeros((*xr.shape[:-1], self.dout2))
        yi = torch.zeros_like(yr)
        for b in range(self.B):
            Hb, Ab = self.H[b], self.As[b]
            u = Hb[0,0]*xr + Hb[0,1]*xi
            v = Hb[1,0]*xr + Hb[1,1]*xi
            yr = yr + u @ Ab.t()
            yi = yi + v @ Ab.t()
        y = torch.stack([yr, yi], dim=-2).reshape(*yr.shape[:-1], 2 * self.dout2)
        if self.bias is not None:
            y = y + self.bias
        return y


class PHMResidual(nn.Module):
    """
    y = (1 - alpha) * dense(x) + alpha * phm(x)
    Caches (dense_out, phm_out) to enable tiny reconstruction loss.
    """
    def __init__(self, dense: nn.Linear, phm: ComplexPHMLinear, alpha_init=0.0):
        super().__init__()
        self.dense = dense
        for p in self.dense.parameters(): p.requires_grad_(False)
        self.phm = phm
        self.register_buffer("alpha", torch.tensor(float(alpha_init)))
        self._cache_dense = None
        self._cache_phm = None
        self._collect_cache = False

    def set_alpha(self, a: float): self.alpha = self.alpha.new_tensor(float(a))

    def forward(self, x):
        d = self.dense(x)
        p = self.phm(x)
        if self.training and self._collect_cache:
            self._cache_dense = d.detach()
            self._cache_phm   = p
        return (1.0 - self.alpha) * d + self.alpha * p


# -------------- helpers for naming --------------
def _is_visual_scope(name: str) -> bool:
    n = name.lower()
    return (".visual." in n) or n.startswith("model.visual") or ("vision_tower" in n)

def _in_attn_scope(name: str) -> bool:
    n = name.lower()
    return (".attn." in n) or (".self_attn." in n) or (".attention." in n)

def _is_qkv_child(child_name: str) -> bool:
    return child_name in {"qkv", "q_proj", "k_proj", "v_proj"}

def _is_ffn_child(child_name: str) -> bool:
    return child_name in {"up_proj", "down_proj", "w1", "w3", "gate_proj"}

def _is_attn_proj(child_name: str) -> bool:
    return child_name in {"o_proj", "out_proj", "proj"}

def _is_excluded_linear_path(name: str) -> bool:
    n = name.lower()
    return ("embed" in n) or ("lm_head" in n)

PROJECTOR_TAGS = ["mm_projector","multimodal_projector","multi_modal_projector",
                  "vision_to_llm","image_projector",".projector","vision_language_projector"]

def _is_projector_path(fq_name: str) -> bool:
    n = fq_name.lower()
    if _is_visual_scope(n):  # projector is outside vision tower
        return False
    return any(tag in n for tag in PROJECTOR_TAGS)

def _should_swap_linear(mod: nn.Linear, size_threshold: int = 65536) -> bool:
    din, dout = mod.in_features, mod.out_features
    if din % 2 != 0 or dout % 2 != 0: return False
    return (din * dout) >= size_threshold


def detect_num_language_layers(model) -> int:
    """
    Try to detect number of language layers by scanning names like:
    'model.language_model.layers.{i}.mlp.*'
    """
    max_idx = -1
    for name, _ in model.named_modules():
        m = re.search(r"model\.language_model\.layers\.(\d+)\.", name)
        if m:
            idx = int(m.group(1))
            if idx > max_idx: max_idx = idx
    return max_idx + 1 if max_idx >= 0 else getattr(getattr(model, "config", SimpleNamespace()), "num_hidden_layers", 0)


def swap_many_linears_to_complex_selectiveB(
    model: nn.Module,
    B_default: int = 2,
    top_lang_layers_B3: int = 0,
    project_from_dense: bool = True,
    include_ffn: bool = True,
    include_misc: bool = False,
    size_threshold: int = 65536,
    force_swap_projector: bool = False,
    verbose: bool = True
) -> int:
    """
    Swap FFN linears to PHMResidual with selective B:
      - language MLP top-K layers -> B=3
      - everything else (eligible) -> B_default (e.g., 2)
    Never touch attention (Q/K/V/out-proj). Keep projector dense unless forced.
    """
    num_swapped = 0
    phm_wrapped = []

    num_lang_layers = detect_num_language_layers(model)
    b3_threshold = max(0, num_lang_layers - top_lang_layers_B3) if top_lang_layers_B3 > 0 else num_lang_layers + 1

    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if not isinstance(child, nn.Linear): continue
            fq = f"{parent_name}.{child_name}" if parent_name else child_name
            if _is_excluded_linear_path(fq): continue

            in_visual  = _is_visual_scope(fq)
            in_attn    = _in_attn_scope(fq)
            is_qkv     = _is_qkv_child(child_name)
            is_ffn     = _is_ffn_child(child_name)
            is_attnprj = _is_attn_proj(child_name)
            is_proj    = _is_projector_path(fq)

            # Never touch attention
            if in_attn and (is_qkv or is_attnprj):
                if verbose: print(f"[skip][ATTN kept dense] {fq}")
                continue
            if (not in_visual) and in_attn:
                if verbose: print(f"[skip][TEXT-ATTN kept dense] {fq}")
                continue

            # Projector optional
            if force_swap_projector and is_proj and _should_swap_linear(child, size_threshold):
                dev, dt = child.weight.device, child.weight.dtype
                phm = ComplexPHMLinear(child.in_features, child.out_features, B=B_default,
                                       bias=(child.bias is not None)).to(dev, dt)
                try:
                    if project_from_dense: phm.project_from_dense(child.weight.data)
                except Exception as e:
                    if verbose: print(f"[warn] projection failed at {fq} (B={B_default}): {e} -> random init kept")
                wrapped = PHMResidual(child, phm, alpha_init=0.0)
                setattr(parent, child_name, wrapped); num_swapped += 1
                phm_wrapped.append(wrapped)
                if verbose: print(f"[swap][PROJECTOR] {fq} -> PHMResidual(ComplexPHMLinear,B={B_default})")
                continue

            # FFNs
            if include_ffn and is_ffn and _should_swap_linear(child, size_threshold):
                # determine layer index if language MLP
                layer_idx = None
                m = re.search(r"model\.language_model\.layers\.(\d+)\.mlp\.", fq)
                if m:
                    layer_idx = int(m.group(1))
                B_here = 3 if (layer_idx is not None and layer_idx >= b3_threshold) else B_default

                dev, dt = child.weight.device, child.weight.dtype
                phm = ComplexPHMLinear(child.in_features, child.out_features, B=B_here,
                                       bias=(child.bias is not None)).to(dev, dt)
                try:
                    if project_from_dense: phm.project_from_dense(child.weight.data)
                except Exception as e:
                    if verbose: print(f"[warn] projection failed at {fq} (B={B_here}): {e} -> random init kept")
                wrapped = PHMResidual(child, phm, alpha_init=0.0)
                setattr(parent, child_name, wrapped); num_swapped += 1
                phm_wrapped.append(wrapped)
                if verbose: print(f"[swap][FFN] {fq} -> PHMResidual(ComplexPHMLinear,B={B_here})")
                continue

            # Misc
            if include_misc and _should_swap_linear(child, size_threshold):
                dev, dt = child.weight.device, child.weight.dtype
                phm = ComplexPHMLinear(child.in_features, child.out_features, B=B_default,
                                       bias=(child.bias is not None)).to(dev, dt)
                try:
                    if project_from_dense: phm.project_from_dense(child.weight.data)
                except Exception as e:
                    if verbose: print(f"[warn] projection failed at {fq} (B={B_default}): {e} -> random init kept")
                wrapped = PHMResidual(child, phm, alpha_init=0.0)
                setattr(parent, child_name, wrapped); num_swapped += 1
                phm_wrapped.append(wrapped)
                if verbose: print(f"[swap][MISC] {fq} -> PHMResidual(ComplexPHMLinear,B={B_default})")
                continue

    model._phm_wrapped = phm_wrapped
    if verbose:
        print(f"Total modules swapped (residual PHM): {num_swapped}")
        print(f"[capacity] num_lang_layers={num_lang_layers}, top_lang_layers_B3={top_lang_layers_B3}")
    return num_swapped


Data pipeline 

In [ ]:
PROMPTS = [
    "Describe this image in one concise sentence.",
    "Write a factual caption for the image.",
    "What is happening in the image?",
]

def expand_flickr8k_split(ds_split):
    def _map(batch):
        images, instrs, resps = [], [], []
        c0, c1, c2, c3, c4 = (
            batch.get("caption_0"), batch.get("caption_1"),
            batch.get("caption_2"), batch.get("caption_3"), batch.get("caption_4"),
        )
        for img, s0, s1, s2, s3, s4 in zip(batch["image"], c0, c1, c2, c3, c4):
            for cap in (s0, s1, s2, s3, s4):
                if cap is None: continue
                images.append(img)
                instrs.append(random.choice(PROMPTS))
                resps.append(cap)
        return {"image": images, "instruction": instrs, "response": resps}

    cols_to_remove = [c for c in ds_split.column_names if c.startswith("caption_")]
    return ds_split.map(_map, batched=True, remove_columns=cols_to_remove)

class Flickr8kHF(Dataset):
    def __init__(self, hf_dataset): self.ds = hf_dataset
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        it = self.ds[idx]
        return {"image": it["image"], "instruction": it["instruction"], "response": it["response"]}

def collate_qwen(batch, processor, cfg, pad_to_multiple_of=8):
    conversations = [
        [
            {"role": "user", "content": [
                {"type": "image", "image": b["image"].convert("RGB")},
                {"type": "text",  "text": b["instruction"].strip()}
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": b["response"].strip()}
            ]},
        ]
        for b in batch
    ]

    full_texts = [processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
                  for conv in conversations]
    prefixes   = [processor.apply_chat_template(conv[:1], tokenize=False, add_generation_prompt=True)
                  for conv in conversations]

    image_inputs, video_inputs = process_vision_info(conversations)

    proc_out = processor(
        text=full_texts, images=image_inputs, videos=video_inputs,
        padding=True, truncation=False, return_tensors="pt", pad_to_multiple_of=pad_to_multiple_of,
    )
    input_ids      = proc_out["input_ids"]
    attention_mask = proc_out["attention_mask"]

    pref_out = processor(
        text=prefixes, images=image_inputs, videos=video_inputs,
        padding=True, truncation=False, return_tensors="pt", pad_to_multiple_of=pad_to_multiple_of,
    )
    prefix_ids = pref_out["input_ids"]

    labels = input_ids.clone(); labels[:] = -100
    pad_id = processor.tokenizer.pad_token_id
    B = input_ids.size(0)
    for i in range(B):
        k = int((prefix_ids[i] != pad_id).sum().item())
        seq_len = int(attention_mask[i].sum().item())
        k = min(k, seq_len)
        labels[i, k:seq_len] = input_ids[i, k:seq_len]

    batch_out = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
    for k in ["pixel_values", "pixel_values_videos", "image_grid_thw", "video_grid_thw", "second_per_grid_ts"]:
        if k in proc_out: batch_out[k] = proc_out[k]
    return batch_out


def prepare_or_load_dataset(cfg: SimpleNamespace):
    if cfg.DO_DATA_PREP:
        print("Loading dataset jxie/flickr8k and expanding...")
        raw = load_dataset("jxie/flickr8k")
        train_exp = expand_flickr8k_split(raw["train"])
        val_exp   = expand_flickr8k_split(raw["validation"])
        test_exp  = expand_flickr8k_split(raw["test"])
        ds = DatasetDict(train=train_exp, validation=val_exp, test=test_exp)
        if cfg.DO_SAVE_DATA:
            os.makedirs(cfg.ds_cache, exist_ok=True)
            ds.save_to_disk(cfg.ds_cache)
            print(f"Saved expanded dataset to {cfg.ds_cache}")
        return ds
    else:
        if os.path.isdir(cfg.ds_cache):
            print(f"Loading expanded dataset from {cfg.ds_cache} ...")
            return load_from_disk(cfg.ds_cache)
        else:
            print("Expanded dataset cache not found; falling back to DO_DATA_PREP=True")
            cfg.DO_DATA_PREP = True
            return prepare_or_load_dataset(cfg)




Load model, swap, add LoRA, data

In [ ]:
set_seed(CFG.seed)

processor = AutoProcessor.from_pretrained(CFG.model_id, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(CFG.model_id, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

if hasattr(processor, "image_processor"):
    ip = processor.image_processor
    if hasattr(ip, "size") and isinstance(ip.size, dict) and "shortest_edge" in ip.size:
        ip.size["shortest_edge"] = CFG.image_short_side

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype_for_load = torch.bfloat16 if torch.cuda.is_available() else torch.float32

load_id = CFG.swap_cache if (CFG.DO_LOAD_FROM_SWAP and os.path.isdir(CFG.swap_cache)) else CFG.model_id
print(f"Loading {'swapped cache' if load_id == CFG.swap_cache else 'base model'}: {load_id}")

model = AutoModelForVision2Seq.from_pretrained(
    load_id, torch_dtype=dtype_for_load, device_map=None, trust_remote_code=True,
)
model.to(device)
model.config.use_cache = False
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except Exception:
    model.gradient_checkpointing_enable()

try:
    from torch.backends.cuda import sdp_kernel
    sdp_kernel(enable_flash=True, enable_mem_efficient=True, enable_math=False)
except Exception:
    pass

if CFG.DO_SWAP and load_id == CFG.model_id:
    print("Counting parameters BEFORE swap ...")
    _ = count_params(model, label="[Before swap]")

    print("Swapping big FFNs to Residual Complex-PHM (selective B; never attention) ...")
    swapped = swap_many_linears_to_complex_selectiveB(
        model,
        B_default=CFG.phm_bases_default,
        top_lang_layers_B3=CFG.top_lang_layers_B3,
        project_from_dense=CFG.PROJECT_FROM_DENSE,
        include_ffn=CFG.include_ffn,
        include_misc=CFG.include_misc,
        size_threshold=CFG.size_threshold,
        force_swap_projector=CFG.FORCE_SWAP_PROJECTOR,
        verbose=True,
    )

    if CFG.LORA_TEXT_QKV:
        num_lora = 0
        for parent_name, parent in list(model.named_modules()):
            for child_name, child in list(parent.named_children()):
                fq = f"{parent_name}.{child_name}" if parent_name else child_name
                if not isinstance(child, nn.Linear): continue
                n = fq.lower()
                in_visual = (".visual." in n) or n.startswith("model.visual") or ("vision_tower" in n)
                in_attn = (".attn." in n) or (".self_attn." in n) or (".attention." in n)
                if in_visual or not in_attn: continue
                if child_name not in {"qkv","q_proj","k_proj","v_proj"}: continue
                wrapped = LoRALinear(child, r=CFG.LORA_R, alpha=CFG.LORA_ALPHA, dropout=CFG.LORA_DROPOUT)\
                         .to(child.weight.device, child.weight.dtype)
                setattr(parent, child_name, wrapped); num_lora += 1
                print(f"[lora][TEXT-QKV] {fq} -> LoRALinear(r={CFG.LORA_R})")
        print(f"Total LoRA-wrapped QKV (text): {num_lora}")

    print("Counting parameters AFTER swap ...")
    _ = count_params(model, label="[After swap]")

    if CFG.DO_SAVE_SWAP:
        os.makedirs(CFG.swap_cache, exist_ok=True)
        model.save_pretrained(CFG.swap_cache)
        tokenizer.save_pretrained(CFG.swap_cache)
        processor.save_pretrained(CFG.swap_cache)
        with open(os.path.join(CFG.swap_cache, "SWAPPED.json"), "w") as f:
            json.dump(
                {
                    "B_default": CFG.phm_bases_default,
                    "top_lang_layers_B3": CFG.top_lang_layers_B3,
                    "project_from_dense": CFG.PROJECT_FROM_DENSE,
                    "size_threshold": CFG.size_threshold,
                    "include_ffn": CFG.include_ffn,
                    "include_attn": False,
                    "include_misc": CFG.include_misc,
                    "force_swap_projector": CFG.FORCE_SWAP_PROJECTOR,
                }, f
            )
        print(f"Saved swapped model to: {CFG.swap_cache}")

# Freeze vision tower (small data)
def _freeze(module):
    for p in module.parameters(): p.requires_grad_(False)
if hasattr(model, "vision_tower"): _freeze(model.vision_tower)
elif hasattr(model, "visual"):     _freeze(model.visual)

# Dataset
ds_all = prepare_or_load_dataset(CFG)
train_ds = Flickr8kHF(ds_all["train"])
val_ds   = Flickr8kHF(ds_all["validation"])

collate = lambda b: collate_qwen(b, processor, CFG, pad_to_multiple_of=8)
train_loader = DataLoader(train_ds, batch_size=CFG.train_micro_batch, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=True, collate_fn=collate,
                          persistent_workers=(CFG.num_workers > 0))
val_loader   = DataLoader(val_ds, batch_size=CFG.train_micro_batch, shuffle=False,
                          num_workers=CFG.num_workers, pin_memory=True, collate_fn=collate,
                          persistent_workers=(CFG.num_workers > 0))

print("Ready: model & data")

Training helpers & metrics

In [ ]:
from qwen_vl_utils import process_vision_info
def collect_dense_projector_params(model):
    phm_param_ids = set()
    for _, m in model.named_modules():
        if isinstance(m, ComplexPHMLinear):
            for p in m.parameters(): phm_param_ids.add(id(p))
        if isinstance(m, PHMResidual):
            for p in m.phm.parameters(): phm_param_ids.add(id(p))
    proj_dense_params = []
    for full_name, p in model.named_parameters():
        if _is_projector_path(full_name) and id(p) not in phm_param_ids:
            proj_dense_params.append(p)
    return proj_dense_params


def build_optim_sched(model, cfg, steps_per_epoch):
    # freeze everything
    for p in model.parameters(): p.requires_grad_(False)

    phm_params, head_params, lora_params = [], [], []
    for _, m in model.named_modules():
        if isinstance(m, PHMResidual):
            for p in m.phm.parameters(): p.requires_grad_(True); phm_params.append(p)
        elif isinstance(m, ComplexPHMLinear):
            for p in m.parameters(): p.requires_grad_(True); phm_params.append(p)
    for n, p in model.named_parameters():
        if "lm_head" in n: p.requires_grad_(True); head_params.append(p)
    for _, m in model.named_modules():
        if isinstance(m, LoRALinear) and m.r > 0:
            m.lora_A.requires_grad_(True); m.lora_B.requires_grad_(True)
            lora_params.extend([m.lora_A, m.lora_B])

    projector_dense_params = collect_dense_projector_params(model)
    for p in projector_dense_params: p.requires_grad_(True)

    phm_params = _uniq_params(phm_params)
    head_params = _uniq_params(head_params)
    lora_params = _uniq_params(lora_params)
    projector_dense_params = _uniq_params(projector_dense_params)

    groups = []
    if phm_params:
        groups.append({"params": phm_params, "lr": cfg.lr_ffn, "weight_decay": cfg.weight_decay})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.lr_head, "weight_decay": cfg.weight_decay})
    if lora_params:
        groups.append({"params": lora_params, "lr": cfg.lr_ffn, "weight_decay": 0.0})
    if projector_dense_params:
        groups.append({"params": projector_dense_params, "lr": cfg.lr_ffn, "weight_decay": cfg.weight_decay})

    optim = torch.optim.AdamW(groups)
    total_steps = cfg.epochs * steps_per_epoch
    warmup = max(cfg.warmup_steps, int(cfg.warmup_ratio * total_steps))
    sched = get_cosine_schedule_with_warmup(optim, warmup, total_steps)
    return optim, sched


def _set_fade_alpha(model, alpha: float):
    if hasattr(model, "_phm_wrapped"):
        for m in model._phm_wrapped: m.set_alpha(alpha)

def _toggle_cache(model, flag: bool):
    if hasattr(model, "_phm_wrapped"):
        for m in model._phm_wrapped: m._collect_cache = flag
        if not flag:
            for m in model._phm_wrapped:
                m._cache_dense = None; m._cache_phm = None

def _recon_loss(model):
    if not hasattr(model, "_phm_wrapped"): return torch.tensor(0.0, device=device)
    loss = None; count = 0
    for m in model._phm_wrapped:
        if (m._cache_dense is not None) and (m._cache_phm is not None):
            term = torch.mean((m._cache_phm - m._cache_dense)**2)
            loss = term if loss is None else (loss + term)
            count += 1
    if count == 0: return torch.tensor(0.0, device=device)
    return loss / count


def export_compact_phm_only(model: nn.Module, out_dir: str, verbose=True):
    # destructively drop dense path
    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, PHMResidual):
                setattr(parent, child_name, child.phm)
                if verbose:
                    print(f"[export] {parent_name}.{child_name}: PHMResidual -> ComplexPHMLinear")
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir)


@torch.no_grad()
def clone_to_phm_only(model: nn.Module) -> nn.Module:
    """
    Non-destructive: deepcopy model and drop dense branches in the copy.
    """
    tmp = copy.deepcopy(model)
    for parent_name, parent in list(tmp.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, PHMResidual):
                setattr(parent, child_name, child.phm)
    return tmp


@torch.no_grad()
def _forward_teacher_same_mode(model, batch, scaler_ctx):
    # set α=0 temporarily (dense teacher) on residual modules
    alphas = []
    if hasattr(model, "_phm_wrapped"):
        for m in model._phm_wrapped:
            alphas.append(m.alpha.clone())
            m.set_alpha(0.0)

    teacher_inp = {k: v for k, v in batch.items() if k != "labels"}
    ctx = scaler_ctx if isinstance(scaler_ctx, contextlib._GeneratorContextManager) else nullcontext()
    with ctx:
        out_t = model(**teacher_inp, output_attentions=False, output_hidden_states=False, use_cache=False)
    teacher_logits = out_t.logits.detach()

    if hasattr(model, "_phm_wrapped"):
        for m, a in zip(model._phm_wrapped, alphas):
            m.set_alpha(float(a.item()))
    return teacher_logits


# -------- Label-smoothed CE on valid tokens --------
def masked_label_smoothing_ce(logits, labels, ignore_index=-100, eps=0.1):
    V = logits.size(-1)
    valid = labels.ne(ignore_index)
    if valid.sum() == 0:
        return logits.new_zeros(())
    logits_v = logits[valid]           # [N,V]
    labels_v = labels[valid]           # [N]
    ls = torch.full_like(logits_v, eps / (V - 1))
    ls.scatter_(1, labels_v.unsqueeze(-1), 1.0 - eps)
    logp = torch.log_softmax(logits_v, dim=-1)
    return -(ls * logp).sum(dim=-1).mean()


# -------- Tiny metrics (BLEU-4 / METEOR-lite / CIDEr-lite) --------
def _tok(s: str) -> List[str]:
    return re.findall(r"[A-Za-z0-9]+", s.lower())

def bleu4(hyps: List[str], refs: List[str]) -> float:
    def ngrams(toks, n):
        return list(zip(*[toks[i:] for i in range(n)]))
    precisions = []
    for n in range(1,5):
        num, den = 0, 0
        for h, r in zip(hyps, refs):
            ht, rt = _tok(h), _tok(r)
            h_ngr = Counter(ngrams(ht, n))
            r_ngr = Counter(ngrams(rt, n))
            overlap = {k: min(v, r_ngr.get(k,0)) for k,v in h_ngr.items()}
            num += sum(overlap.values()); den += max(1, sum(h_ngr.values()))
        precisions.append(num / max(1, den))
    # brevity penalty
    hyp_len = sum(len(_tok(h)) for h in hyps)
    ref_len = sum(len(_tok(r)) for r in refs)
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(1, hyp_len))
    # geometric mean
    gm = math.exp(sum((0.25)*math.log(max(1e-9,p)) for p in precisions))
    return bp * gm

def meteor_lite(hyps: List[str], refs: List[str]) -> float:
    # simple unigram F1 with slight recall bias
    scores = []
    for h, r in zip(hyps, refs):
        H, R = Counter(_tok(h)), Counter(_tok(r))
        overlap = sum(min(H[w], R[w]) for w in set(H)|set(R))
        p = overlap / max(1, sum(H.values()))
        q = overlap / max(1, sum(R.values()))
        if (p+q)==0: scores.append(0.0); continue
        # harmonic mean with recall bias
        Fmean = (10*p*q) / (9*p + q + 1e-9)
        scores.append(Fmean)
    return sum(scores)/max(1,len(scores))

def cider_lite(hyps: List[str], refs: List[str], n_max=4) -> float:
    # crude TF-IDF n-gram cosine similarity
    N = len(hyps)
    df = [defaultdict(int) for _ in range(n_max)]
    ref_ngrams_all = []
    for r in refs:
        rt = _tok(r)
        ref_ngrams = []
        for n in range(1, n_max+1):
            items = list(zip(*[rt[i:] for i in range(n)]))
            ref_ngrams.append(items)
            for g in set(items): df[n-1][g] += 1
        ref_ngrams_all.append(ref_ngrams)
    def vec(toks):
        vecs = []
        for n in range(1, n_max+1):
            items = list(zip(*[toks[i:] for i in range(n)]))
            c = Counter(items)
            v = {}
            for g, cnt in c.items():
                idf = math.log((N + 1.0) / (df[n-1].get(g,1)))
                v[g] = cnt * idf
            vecs.append(v)
        return vecs
    scores = []
    for h, r_ngrams in zip(hyps, ref_ngrams_all):
        hv = vec(_tok(h))
        # reference vector
        rv = []
        for n in range(n_max):
            c = Counter(r_ngrams[n])
            v = {}
            for g, cnt in c.items():
                idf = math.log((N + 1.0) / (df[n].get(g,1)))
                v[g] = cnt * idf
            rv.append(v)
        # cosine per n
        sim = 0.0
        for n in range(n_max):
            dot = sum(hv[n].get(g,0.0)*rv[n].get(g,0.0) for g in set(hv[n])|set(rv[n]))
            nh = math.sqrt(sum(v*v for v in hv[n].values())) + 1e-9
            nr = math.sqrt(sum(v*v for v in rv[n].values())) + 1e-9
            sim += dot/(nh*nr)
        scores.append((10.0/n_max)*sim)  # scale like CIDEr
    return sum(scores)/max(1,len(scores))


# ==== Eval helpers (no globals; always returns float) ====
@torch.no_grad()
def evaluate_ce(model, loader, device, autocast_ctx, eps=None):
    # default: match training CE unless overridden
    if eps is None:
        eps = getattr(CFG, "CE_LABEL_SMOOTH", 0.1)

    model.eval()
    tot, cnt = 0.0, 0
    for vb in loader:
        vb = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in vb.items()}
        with autocast_ctx:
            out = model(**vb)
        ce_val = masked_label_smoothing_ce(out.logits.float(), vb["labels"], eps=eps)
        bs = vb["labels"].size(0)
        tot += ce_val.item() * bs
        cnt += bs
    return float(tot / max(1, cnt))


@torch.no_grad()
def evaluate_caption_metrics(model, processor, dataset, device, max_samples=256):
    model.eval()
    preds, refs = [], []
    with torch.no_grad():
        for i in range(min(max_samples, len(dataset))):
            ex = dataset[i]
            image = ex["image"].convert("RGB")
            instr = ex["instruction"]
            ref   = ex["response"]

            conv = [
                {"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": instr}
                ]}
            ]
            prompt = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)

            # Let the helper build exactly-matching multimodal inputs
            image_inputs, video_inputs = process_vision_info([conv])

            proc = processor(
                text=[prompt],
                images=image_inputs,
                videos=video_inputs,          # will be None for image-only
                return_tensors="pt",
                padding=True
            )
            proc = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in proc.items()}

            gen_ids = model.generate(**proc, max_new_tokens=64, do_sample=False)
            pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

            preds.append(pred)
            refs.append(ref)
            b4 = bleu4(preds, refs)
            me = meteor_lite(preds, refs)
            ci = cider_lite(preds, refs)

    # ... compute BLEU4 / METEOR / CIDEr on (preds, refs) as you already do ...
    return {"BLEU4": b4, "METEOR": me, "CIDEr": ci}



In [8]:
# ==== FLOPs utils (PHM-only, attention-aware, hook-based) ====
def _numel_prefix(x: torch.Tensor, last_dim: int) -> int:
    # how many "vectors" of length last_dim are in x
    return int(x.numel() // max(1, last_dim))

def add_flops_counters_for_phm_model(model: nn.Module, count_activations: bool = False):
    """
    Registers forward hooks to count FLOPs for:
    - ComplexPHMLinear (dominant cost: 2 GEMMs per base)
    - nn.Linear (dense projections)
    - Attention matmuls (QK^T + softmax + AV) best-effort
    Optionally add cheap elementwise activation costs.
    Returns (totals, hooks).
    """
    totals = {"flops": 0}
    hooks = []

    def _acc(n):
        # accumulate in Python int to avoid overflow in long runs
        totals["flops"] = int(totals["flops"]) + int(n)

    def linear_hook(mod: nn.Linear, inp, out):
        x = inp[0]
        if not torch.is_tensor(x) or x.dim() == 0: return
        N = _numel_prefix(x, mod.in_features)
        _acc(2 * N * mod.in_features * mod.out_features)  # mult+add

    def phm_hook(mod, inp, out):
        x = inp[0]
        if not torch.is_tensor(x) or x.dim() == 0: return
        din, din2, dout2, B = mod.din, mod.din2, mod.dout2, mod.B
        N = _numel_prefix(x, din)
        gemm = 4 * N * din2 * dout2          # two GEMMs per base, each 2*N*din2*dout2
        mix  = 6 * N * din2                   # u/v linear combs (approx)
        acc  = 2 * N * dout2                  # yr/yi accum adds (approx)
        _acc(B * (gemm + mix + acc))

    def attn_hook(mod, inp, out):
        # Best-effort FLOPs for scaled dot-product attention:
        # QK^T: 2 * B * nh * T * T * d
        # softmax ~ B * nh * T * T
        # AV:     2 * B * nh * T * T * d
        x = inp[0]
        if not torch.is_tensor(x) or x.dim() < 3: return
        try:
            Bsz, T, H = int(x.shape[-3]), int(x.shape[-2]), int(x.shape[-1])
            nh = getattr(mod, "num_heads", None) or getattr(mod, "num_attention_heads", None)
            if nh is None or nh == 0: return
            hd = getattr(mod, "head_dim", None) or (H // nh if H % nh == 0 else None)
            if hd is None or hd == 0: return
            _acc(2 * Bsz * nh * T * T * hd)   # QK^T
            _acc(    Bsz * nh * T * T)        # softmax (approx)
            _acc(2 * Bsz * nh * T * T * hd)   # AV
        except Exception:
            return

    def act_hook(mod, inp, out):
        # Very rough cost for elementwise activation (e.g., SiLU/GELU)
        if not count_activations: return
        x = out if torch.is_tensor(out) else (out[0] if isinstance(out, (tuple, list)) and torch.is_tensor(out[0]) else None)
        if x is None: return
        _acc(int(x.numel()))  # ~1 flop/elem (very approximate)

    for name, m in model.named_modules():
        if isinstance(m, ComplexPHMLinear):
            hooks.append(m.register_forward_hook(phm_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
        elif hasattr(m, "forward") and (hasattr(m, "num_heads") or hasattr(m, "num_attention_heads")):
            hooks.append(m.register_forward_hook(attn_hook))
        # (Optional) count activations in MLPs
        if count_activations and hasattr(nn, "SiLU") and isinstance(m, nn.SiLU):
            hooks.append(m.register_forward_hook(act_hook))
        if count_activations and hasattr(nn, "GELU") and isinstance(m, nn.GELU):
            hooks.append(m.register_forward_hook(act_hook))

    return totals, hooks

@torch.no_grad()
def measure_validation_flops_phm(model: nn.Module, val_loader, device, scaler_ctx, max_batches: int = 5, count_activations: bool = False):
    """
    Runs a few validation batches with hooks attached and returns:
      total_flops, avg_gflops_per_batch, num_measured_batches
    """
    model.eval()
    totals, hooks = add_flops_counters_for_phm_model(model, count_activations=count_activations)
    seen = 0
    try:
        for i, batch in enumerate(val_loader):
            batch = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}
            with scaler_ctx:
                _ = model(**batch)
            seen += 1
            if seen >= max_batches: break
    finally:
        for h in hooks: 
            try: h.remove()
            except Exception: pass

    gflops_per_batch = (float(totals["flops"]) / 1e9) / max(1, seen)
    return int(totals["flops"]), gflops_per_batch, seen


In [9]:
# ---- resume scheduler position (best-effort) ----
steps_per_epoch = len(train_loader)
resume_ep  = getattr(CFG, "RESUME_EPOCH", 0) or 0
resume_stp = getattr(CFG, "RESUME_STEP", 0) or 0
resume_global = resume_ep * steps_per_epoch + resume_stp
if resume_global > 0:
    try:
        sched.last_epoch = resume_global - 1
        # sync current LR immediately
        last_lrs = sched.get_last_lr()
        for g, lr in zip(optim.param_groups, last_lrs):
            g["lr"] = lr
    except Exception:
        pass
    # make alpha ramp continue from the same global step
    global_step = resume_global
else:
    global_step = 0

In [11]:
def resume_from_preval_ckpt(ckpt_path, model, optim, sched, device):
    print(f"[ckpt] loading {ckpt_path}")
    state = torch.load(ckpt_path, map_location=device)

    # weights first
    model.load_state_dict(state["model_state"], strict=False)

    # optimizer/scheduler (best-effort)
    if optim is not None and "optim_state" in state:
        try: optim.load_state_dict(state["optim_state"])
        except Exception as e: print(f"[ckpt][warn] optim load skipped: {e}")
    if sched is not None and "sched_state" in state:
        try: sched.load_state_dict(state["sched_state"])
        except Exception as e: print(f"[ckpt][warn] sched load skipped: {e}")

    # resume positions (we saved right *after* last train step of the epoch)
    resume_ep   = int(state.get("epoch", 0))
    resume_stp  = int(state.get("step", -1)) + 1   # start at next step
    global_step = int(state.get("global_step", 0))

    # RNG restore (optional)
    rng = state.get("rng", None)
    if rng:
        try:
            import numpy as _np, random as _random
            torch.set_rng_state(rng["torch"])
            _random.setstate(rng["py_random"])
            if torch.cuda.is_available():
                if "cuda_all" in rng: torch.cuda.set_rng_state_all(rng["cuda_all"])
                elif "cuda" in rng:   torch.cuda.set_rng_state(rng["cuda"])
            if "numpy" in rng: _np.random.set_state(rng["numpy"])
        except Exception as e:
            print(f"[ckpt][warn] RNG restore skipped: {e}")

    print(f"[ckpt] will resume at epoch={resume_ep}, step={resume_stp}, global_step={global_step}")
    return resume_ep, resume_stp, global_step


Train (Stage A & B)

In [ ]:
import numpy as np

def _rng_snapshot():
    snap = {
        "torch": torch.get_rng_state(),
        "py_random": random.getstate(),
    }
    if torch.cuda.is_available():
        try:
            snap["cuda_all"] = torch.cuda.get_rng_state_all()
        except Exception:
            snap["cuda"] = torch.cuda.get_rng_state()
    try:
        snap["numpy"] = np.random.get_state()
    except Exception:
        pass
    return snap

def _save_preval_ckpt(path, model, optim, sched, epoch, step, global_step, cfg, best_metric_val, best_epoch):
    state = {
        "epoch": int(epoch),
        "step": int(step),
        "global_step": int(global_step),
        "model_state": model.state_dict(),
        "optim_state": optim.state_dict(),
        "sched_state": sched.state_dict(),
        "best_metric_val": (None if best_metric_val is None else float(best_metric_val)),
        "best_epoch": int(best_epoch),
        "cfg": vars(cfg),
        "rng": _rng_snapshot(),
    }
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)
    print(f"[ckpt] saved pre-val checkpoint -> {path}")

def train(model, processor, tokenizer, cfg):
    use_cuda = torch.cuda.is_available()
    if use_cuda: torch.backends.cuda.matmul.allow_tf32 = True
    scaler_ctx = (torch.autocast(device_type="cuda", dtype=torch.bfloat16) if use_cuda else contextlib.nullcontext())

    os.makedirs(cfg.out_dir, exist_ok=True)
    optim, sched = build_optim_sched(model, cfg, len(train_loader))

    accum = max(1, cfg.grad_accum)
    total_steps = cfg.epochs * len(train_loader)
    fade_steps = max(1, int(cfg.PHM_FADE_FRACTION * total_steps))

    # ---- resume controls (safe defaults) ----
    resume_ep  = int(getattr(cfg, "resume_ep", 0))
    resume_stp = int(getattr(cfg, "resume_stp", 0))
    global_step = int(getattr(cfg, "resume_global_step", 0))

    best_metric_val = None
    best_epoch = -1
    best_ckpt_dir = None

    print("Starting Stage A (residual PHM) ...")
    for ep in range(cfg.epochs):
        if ep < resume_ep:
            # keep alpha ramp consistent while skipping whole epochs
            steps_to_skip = len(train_loader)
            global_step += steps_to_skip
            continue

        model.train()
        optim.zero_grad(set_to_none=True)

        # -------- Train epoch (single loop; skip to resume_stp if resuming mid-epoch) --------
        for step, batch in enumerate(train_loader):
            if ep == resume_ep and step < resume_stp:
                # keep alpha ramp consistent
                alpha = min(1.0, global_step / float(fade_steps))
                _set_fade_alpha(model, alpha)
                global_step += 1
                continue

            alpha = min(1.0, global_step / float(fade_steps))
            _set_fade_alpha(model, alpha)

            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            _toggle_cache(model, True)

            # KD teacher (α=0 dense) if enabled
            teacher_logits = None
            if cfg.KD_ENABLE and (global_step % cfg.KD_EVERY_N == 0):
                try:
                    teacher_logits = _forward_teacher_same_mode(model, batch, scaler_ctx)
                except ValueError:
                    teacher_logits = None

            with scaler_ctx:
                out = model(**batch)
            logits_s, labels = out.logits, batch["labels"]
            ce = masked_label_smoothing_ce(logits_s.float(), labels, eps=cfg.CE_LABEL_SMOOTH)

            kd = torch.tensor(0.0, device=device)
            if teacher_logits is not None:
                valid = labels.ne(-100)
                s_flat = (logits_s.float()[valid] / cfg.KD_T)
                t_flat = (teacher_logits.float()[valid] / cfg.KD_T)
                kd = F.kl_div(
                    F.log_softmax(s_flat, dim=-1),
                    F.softmax(t_flat, dim=-1),
                    reduction="batchmean",
                ) * (cfg.KD_T ** 2)

            # KD lambda ramp
            lam_max = cfg.KD_LAMBDA
            lam = lam_max * min(1.0, global_step / float(fade_steps))

            recon = _recon_loss(model) * cfg.PHM_RECON_WEIGHT
            loss = ((1.0 - lam) * ce + lam * kd + recon) / accum

            loss.backward()
            _toggle_cache(model, False)

            if (step + 1) % accum == 0:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
                optim.step(); sched.step(); optim.zero_grad(set_to_none=True)
                gc.collect()
                if use_cuda: torch.cuda.empty_cache()

            if step % cfg.log_every == 0:
                print(f"epoch {ep} step {step}/{len(train_loader)} "
                      f"loss {(loss.item()*accum):.4f}  ce {ce.item():.4f}  kd {kd.item():.4f} "
                      f"recon {recon.item():.6f}  α={alpha:.2f}")

            global_step += 1

        # -------- SAVE CHECKPOINT ***BEFORE*** VALIDATION --------
        ckpt_dir = os.path.join(cfg.out_dir, "ckpts")
        ckpt_path = os.path.join(ckpt_dir, f"ep{ep:03d}_preval.pt")
        _save_preval_ckpt(
            ckpt_path, model, optim, sched,
            epoch=ep, step=(len(train_loader)-1), global_step=global_step,
            cfg=cfg, best_metric_val=best_metric_val, best_epoch=best_epoch
        )

        # -------- Validation (student CE) --------
        val_ce_student = evaluate_ce(model, val_loader, device, scaler_ctx)
        print(f"[val][StageA][student] epoch {ep} CE: {val_ce_student:.4f}")

        # -------- PHM-only validation (clone drop-dense) --------
        torch.cuda.empty_cache()
        phm_only = clone_to_phm_only(model).to(device)
        phm_only.config.use_cache = False
        ce_phm = evaluate_ce(phm_only, val_loader, device, scaler_ctx)
        print(f"[val][StageA][PHM-only] epoch {ep} CE: {ce_phm:.4f}")

        # caption metrics on subset
        metrics = evaluate_caption_metrics(phm_only, processor, val_ds, device,
                                           max_samples=cfg.VAL_MAX_CAPTION_SAMPLES)
        print(f"[val][StageA][PHM-only] BLEU4={metrics['BLEU4']:.4f}  METEOR={metrics['METEOR']:.4f}  CIDEr={metrics['CIDEr']:.4f}")

        # -------- FLOPs on PHM-only (few batches) --------
        avg_gflops, nb, tot_flops = None, 0, None
        try:
            FLOPS_SAMPLE_STEPS = getattr(cfg, "FLOPS_SAMPLE_STEPS", 5)
            tot_flops, avg_gflops, nb = measure_validation_flops_phm(
                phm_only, val_loader, device, scaler_ctx, max_batches=FLOPS_SAMPLE_STEPS
            )
            print(f"[val][StageA][PHM-only][FLOPs] avg over {nb} batches: {avg_gflops:.2f} GFLOPs/batch "
                  f"(total {tot_flops/1e12:.3f} TFLOPs for sampled)")
        except Exception as e:
            print(f"[val][StageA][PHM-only][FLOPs] skipped: {e}")

        # -------- Save "best" based on selection metric --------
        if cfg.selection_metric == "ce_phm":
            metric_val = ce_phm; better = (best_metric_val is None) or (metric_val < best_metric_val - 1e-6)
        else:  # "cider"
            metric_val = metrics["CIDEr"]; better = (best_metric_val is None) or (metric_val > best_metric_val + 1e-6)

        if better:
            best_metric_val = metric_val
            best_epoch = ep
            best_ckpt_dir = os.path.join(cfg.out_dir, "best_stageA_phmonly")
            # save PHM-only (compact) + tokenizer/processor
            if cfg.LORA_MERGE_ON_SAVE: merge_all_lora_(phm_only)
            os.makedirs(best_ckpt_dir, exist_ok=True)
            phm_only.save_pretrained(best_ckpt_dir)
            tokenizer.save_pretrained(best_ckpt_dir)
            processor.save_pretrained(best_ckpt_dir)
            with open(os.path.join(cfg.out_dir, "BEST_STAGEA.json"), "w") as f:
                json.dump({
                    "epoch": ep,
                    "selection_metric": cfg.selection_metric,
                    "metric_value": float(metric_val),
                    "ce_phm": float(ce_phm),
                    "BLEU4": float(metrics["BLEU4"]),
                    "METEOR": float(metrics["METEOR"]),
                    "CIDEr": float(metrics["CIDEr"]),
                    "avg_GFLOPs_per_batch": (float(avg_gflops) if avg_gflops is not None else None),
                    "flops_sampled_batches": int(nb) if nb else 0,
                }, f, indent=2)
            print(f"[save] New best PHM-only ({cfg.selection_metric}={metric_val:.4f}) -> {best_ckpt_dir}")

        del phm_only
        torch.cuda.empty_cache()

    print(f"Stage A done. Best epoch={best_epoch}, best metric ({cfg.selection_metric})={best_metric_val}")

    # =========================
    # Stage B (PHM-only, short)
    # =========================
    if cfg.STAGE_B_ENABLE and best_ckpt_dir is not None:
        print("\n=== Stage B (PHM-only finetune) ===")
        # Load the best PHM-only checkpoint
        phm_only = AutoModelForVision2Seq.from_pretrained(best_ckpt_dir, torch_dtype=dtype_for_load, trust_remote_code=True)
        phm_only.to(device); phm_only.config.use_cache = False

        # ---- BEFORE finetune: run a validation pass (as requested) ----
        base_ce = evaluate_ce(phm_only, val_loader, device, scaler_ctx)
        base_metrics = evaluate_caption_metrics(phm_only, processor, val_ds, device,
                                                max_samples=cfg.VAL_MAX_CAPTION_SAMPLES)
        print(f"[val][StageB-start][PHM-only] CE={base_ce:.4f}  "
              f"BLEU4={base_metrics['BLEU4']:.4f}  METEOR={base_metrics['METEOR']:.4f}  CIDEr={base_metrics['CIDEr']:.4f}")

        # build simple optimizer/sched for PHM-only
        for p in phm_only.parameters(): p.requires_grad_(False)
        tune_params = []
        for _, m in phm_only.named_modules():
            if isinstance(m, ComplexPHMLinear):
                for p in m.parameters(): p.requires_grad_(True); tune_params.append(p)
        for n, p in phm_only.named_parameters():
            if "lm_head" in n:
                p.requires_grad_(True); tune_params.append(p)
        tune_params = _uniq_params(tune_params)

        optimB = torch.optim.AdamW(
            [{"params": tune_params, "lr": cfg.stageB_lr, "weight_decay": cfg.stageB_wd}]
        )
        total_stepsB = cfg.stageB_epochs * len(train_loader)
        warmupB = int(cfg.stageB_warmup_ratio * total_stepsB)
        schedB = get_cosine_schedule_with_warmup(optimB, warmupB, total_stepsB)
        accumB = max(1, cfg.grad_accum)

        # train α=1, no recon
        for epB in range(cfg.stageB_epochs):
            phm_only.train()
            for step, batch in enumerate(train_loader):
                batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                with scaler_ctx:
                    out = phm_only(**batch)
                logits_s, labels = out.logits, batch["labels"]
                ce = masked_label_smoothing_ce(logits_s.float(), labels, eps=cfg.stageB_label_smooth)

                kd = torch.tensor(0.0, device=device)
                if cfg.stageB_KD_ENABLE:
                    pass  # (optional KD teacher)

                loss = (ce + kd) / accumB
                loss.backward()

                if (step + 1) % accumB == 0:
                    torch.nn.utils.clip_grad_norm_([p for p in phm_only.parameters() if p.requires_grad], cfg.grad_clip)
                    optimB.step(); schedB.step(); optimB.zero_grad(set_to_none=True)

                if step % cfg.log_every == 0:
                    print(f"[StageB] ep {epB} step {step}/{len(train_loader)} "
                          f"loss {(loss.item()*accumB):.4f}  ce {ce.item():.4f}")

        # Final validation after Stage B
        final_ce = evaluate_ce(phm_only, val_loader, device, scaler_ctx)
        final_metrics = evaluate_caption_metrics(phm_only, processor, val_ds, device,
                                                 max_samples=cfg.VAL_MAX_CAPTION_SAMPLES)
        print(f"[val][StageB-end][PHM-only] CE={final_ce:.4f}  "
              f"BLEU4={final_metrics['BLEU4']:.4f}  METEOR={final_metrics['METEOR']:.4f}  CIDEr={final_metrics['CIDEr']:.4f}")

        # Save Stage B final
        final_dir = os.path.join(cfg.out_dir, "final_stageB_phmonly")
        os.makedirs(final_dir, exist_ok=True)
        phm_only.save_pretrained(final_dir)
        tokenizer.save_pretrained(final_dir)
        processor.save_pretrained(final_dir)
        with open(os.path.join(final_dir, "VAL.json"), "w") as f:
            json.dump({
                "stageB_epochs": cfg.stageB_epochs,
                "CE": float(final_ce),
                "BLEU4": float(final_metrics["BLEU4"]),
                "METEOR": float(final_metrics["METEOR"]),
                "CIDEr": float(final_metrics["CIDEr"]),
            }, f, indent=2)
        print(f"[save] Stage B PHM-only saved to {final_dir}")

    print("Training complete.")


 Run

In [ ]:

set_seed(CFG.seed)
train(model, processor, tokenizer, CFG)